In [1]:
%uv pip install huggingface_hub
%uv pip install timm
#!pip install gdown
%uv pip install --upgrade transformers
%uv pip install accelerate
%uv pip install datasets


Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 21ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 40 packages in 215ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 104ms
Installed 1 package in 168ms
 + timm==1.0.27
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 27 packages in 183m

In [ ]:
# import zipfile
# import os

# zip_file_path = 'subset_data.zip' 

# extraction_dir = '.'

# if not os.path.exists(extraction_dir):
#     os.makedirs(extraction_dir)

# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     zip_ref.extractall(extraction_dir)

# print(f"'{zip_file_path}' extracted to '{extraction_dir}' successfully!")

'subset_data.zip' extracted to '.' successfully!


In [ ]:
from huggingface_hub import login

login(token="")

In [ ]:
import os
import torch
import gc
import json

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [7]:
#pip install -U "transformers>=4.51.0"

In [3]:
from datasets import load_dataset

dataset = load_dataset('ranwakhaled/EgyM3AV')
dataset = dataset['test']

README.md:   0%|          | 0.00/668 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/447M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/409M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/110M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/97.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10321 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1297 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1067 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
import torch
from tqdm import tqdm
from collections import defaultdict
import pandas as pd

model_id = "RanaGaber/gemma-3-12b-Full-IT-EGY-1E"
model_name = model_id.split("/")[-1]

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
).eval()

processor = AutoProcessor.from_pretrained(model_id)

BATCH_SIZE = 8

PROMPT_TEMPLATE = (
    "Explain this slide in **Egyptian Arabic** using **clear** and **simple** sentences.\n"
    "Cover all visible elements in the slides including text, formulas, images and diagrams.\n"
    "Do NOT add new information, examples, definitions, or assumptions "
    "that are not explicitly shown on the slide."
)

prompt_str = f"""<start_of_image>
User: {PROMPT_TEMPLATE}
Assistant:"""


lecture_to_indices = defaultdict(list)

for idx, row in enumerate(dataset):
    lecture_to_indices[row["lecture"]].append(idx)

records = []

for lec, indices in tqdm(lecture_to_indices.items(), desc="Lectures"):

    batch_images = []
    batch_rows = []

    for idx in indices:

        row = dataset[idx]

        image = row["image"]

        if image.mode != "RGB":
            image = image.convert("RGB")

        batch_images.append(image)
        batch_rows.append(row)

        if len(batch_images) == BATCH_SIZE:

            inputs = processor(
            images=[[img] for img in batch_images],
            text=[prompt_str] * len(batch_images),
            return_tensors="pt",
            padding=True,
            truncation=False,)

            inputs = {
                k: v.to(model.device)
                for k, v in inputs.items()
            }

            with torch.inference_mode():

                generations = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,

                    repetition_penalty=1.2,
                    no_repeat_ngram_size=3,

                    eos_token_id=processor.tokenizer.eos_token_id,)

            input_lengths = (
                inputs["attention_mask"]
                .sum(dim=1)
                .tolist()
            )

            decoded_outputs = []

            for gen, in_len in zip(generations, input_lengths):

                output_tokens = gen[in_len:]

                decoded = processor.decode(
                    output_tokens,
                    skip_special_tokens=True
                )

                decoded_outputs.append(decoded)

            for row_data, decoded in zip(batch_rows, decoded_outputs):

                records.append({
                    "lecture": lec,
                    "image_name": row_data["image_name"],
                    "output": decoded,
                })

            batch_images = []
            batch_rows = []

    if batch_images:

        inputs = processor(
        images=[[img] for img in batch_images],
        text=[prompt_str] * len(batch_images),
        return_tensors="pt",
        padding=True,
        truncation=False,)

        inputs = {
            k: v.to(model.device)
            for k, v in inputs.items()
        }

        with torch.inference_mode():

            generations = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,

                    repetition_penalty=1.2,
                    no_repeat_ngram_size=3,

                    eos_token_id=processor.tokenizer.eos_token_id,)

        input_lengths = (
            inputs["attention_mask"]
            .sum(dim=1)
            .tolist()
        )

        for gen, in_len, row_data in zip(
            generations,
            input_lengths,
            batch_rows
        ):

            output_tokens = gen[in_len:]

            decoded = processor.decode(
                output_tokens,
                skip_special_tokens=True
            )

            records.append({
                "lecture": lec,
                "image_name": row_data["image_name"],
                "output": decoded,
            })

df = pd.DataFrame(records)

config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Lectures:   0%|                                                                         | 0/71 [00:00<?, ?it/s]/usr/local/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Lectures:  23%|██████████████▍                                                 | 16/71 [08:40<27:48, 30.33s/it]

In [ ]:

df.to_csv(
    f"{model_name}_outputs_fast.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head())

In [16]:
df.to_csv(f"{model_name}_outputs.csv", index=False, encoding="utf-8-sig")
print(df.head())

     lecture             image_name  \
0  CHI-91486  CHI-91486-0010000.jpg   
1  CHI-91486  CHI-91486-0047450.jpg   
2  CHI-91486  CHI-91486-0077450.jpg   
3  CHI-91486  CHI-91486-0111450.jpg   
4  CHI-91486  CHI-91486-0123450.jpg   

                                              output  
0  اهو دي اول شريحه في الشرح بتاعنا النهارده الشر...  
1                      عايزني اشرحلك الشريحه دي؟ طيب  
2             عايزني اشرحلك الشريحه دي؟ طيب اسمع مني  
3             عايزني اشرحلك الشريحه دي؟ طيب اسمع مني  
4  عايزني اشرحلك الشريحه دي؟ طيب الشريحه دي بتتكل...  


In [15]:
len(df['image_name'].unique())

1067